In [ ]:
# ===== NB07 | CELL 1: inspect your saved B1 and B2 checkpoints =====
# WHAT: list the files and sizes in each checkpoint folder and print the experiment logs from notebook 04.
# WHY:  confirms the weights really exist (we lost some before) and tells us whether RoBERTa (B2) finished.
from google.colab import drive
drive.mount('/content/drive')
import os, json
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
CK = f'{PROJECT_DIR}/models/saved_checkpoints'

for name in ['B1_distilbert_bce_best', 'B2_roberta_bce_best']:
    p = f'{CK}/{name}'
    if os.path.isdir(p):
        print(name, {f: round(os.path.getsize(f'{p}/{f}') / 1e6, 1) for f in sorted(os.listdir(p))}, '(MB)')
    else:
        print(name, 'MISSING')
for name in ['B1_distilbert_bce', 'B2_roberta_bce']:
    p = f'{PROJECT_DIR}/results/experiment_logs/{name}.json'
    print(f'\n--- log {name}:', 'found' if os.path.exists(p) else 'NOT FOUND')
    if os.path.exists(p):
        print(json.dumps(json.load(open(p)), indent=1)[:1800])
print('CELL 1 DONE')

In [ ]:
# ===== NB07 | CELL 2: write the transformer prediction helper =====
# WHAT: save nnlib.py (fast prediction with fp16 and length-sorted batches) into the src folder.
# WHY:  every neural notebook uses the same function, so predictions are made the same way every time.
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
CODE = r'''"""Transformer helpers shared by every neural notebook (fast prediction)."""
import os
import numpy as np


def predict_with(model, tok, texts, max_len, batch_size=64):
    """Probabilities for a list of texts. Texts are sorted by length, so short comments are not padded to max_len."""
    import torch
    device = next(model.parameters()).device
    use_amp = device.type == 'cuda'
    texts = [t if isinstance(t, str) else '' for t in texts]
    ids = tok(texts, truncation=True, max_length=max_len)['input_ids']
    order = np.argsort([len(x) for x in ids], kind='stable')
    out = np.zeros((len(texts), model.config.num_labels), dtype='float32')
    was_training = model.training
    model.eval()
    with torch.no_grad():
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            b = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt')
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**b).logits
            out[idx] = torch.sigmoid(logits.float()).cpu().numpy()
    if was_training:
        model.train()
    return out


def predict_probs(model_dir, texts, max_len, batch_size=64, fallback_tokenizer=None):
    """Load a saved checkpoint folder and return probabilities for the texts."""
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    try:
        tok = AutoTokenizer.from_pretrained(model_dir)
    except Exception:
        tok = AutoTokenizer.from_pretrained(fallback_tokenizer)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    P = predict_with(model, tok, texts, max_len, batch_size)
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()
    return P
'''
import os, sys, importlib
open(f'{PROJECT_DIR}/src/nnlib.py', 'w').write(CODE)
sys.path.insert(0, f'{PROJECT_DIR}/src')
import nnlib; importlib.reload(nnlib)
print('nnlib written and imported.')
print('CELL 2 DONE')

In [ ]:
# ===== NB07 | CELL 3: predictions from your epoch-3 B1 checkpoint (256 tokens) =====
# WHAT: run your saved DistilBERT on validation, internal test and external test and save the probabilities.
# WHY:  notebook 04 never saved them. The run id is B1_256 because it used 256 tokens (our other runs use 128).
#       Test predictions are SAVED but not opened until the freeze.
from google.colab import drive
drive.mount('/content/drive')
import sys, importlib, time, json
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, nnlib; importlib.reload(toxlib); importlib.reload(nnlib)
from toxlib import LABELS, SPLITS, load_split, load_probs, save_probs, macro_ap
from nnlib import predict_probs
CKPT = f'{PROJECT_DIR}/models/saved_checkpoints/B1_distilbert_bce_best'

t0 = time.time()
for split in SPLITS:
    df = load_split(PROJECT_DIR, split)
    P = predict_probs(CKPT, df.comment_text.tolist(), max_len=256, batch_size=64,
                      fallback_tokenizer='distilbert-base-uncased')
    save_probs(PROJECT_DIR, 'B1_256', split, P)
    print(f'B1_256 {split:14s} saved {P.shape}  ({time.time() - t0:.0f}s)')

# sanity check: do we reproduce the score logged in notebook 04?
Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
logged = json.load(open(f'{PROJECT_DIR}/results/experiment_logs/B1_distilbert_bce.json'))['best_validation_macro_ap']
now = macro_ap(Yva, load_probs(PROJECT_DIR, 'B1_256', 'val'))
print(f'\nSanity check: logged Macro AP {logged:.4f} vs recomputed {now:.4f} ->',
      'OK' if abs(logged - now) < 0.003 else 'CHECK THIS')
print('CELL 3 DONE')

In [ ]:
# ===== NB07 | CELL 4: B1 vs the baselines (validation only) =====
# WHAT: tune thresholds on validation and compare A1, A2 and B1_256, overall and per label.
# WHY:  the first proof that the transformer adds value over the classical baselines.
import sys, importlib
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib; importlib.reload(toxlib)
from toxlib import LABELS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report, log_run
import numpy as np, pandas as pd

Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
rows, per_ap = [], {}
for run in ['A1', 'A2', 'B1_256']:
    P = load_probs(PROJECT_DIR, run, 'val')
    thr = tune_thresholds(Yva, P)
    rep = per_label_report(Yva, P, thr)
    per_ap[run] = rep.set_index('label').AP
    rows.append({'run': run, 'Macro AP': round(macro_ap(Yva, P), 4), 'Macro F1 tuned': round(rep.F1.mean(), 4)})
    if run == 'B1_256':
        print('B1_256 per-label validation results (tuned thresholds):'); print(rep.to_string(index=False))
        log_run(PROJECT_DIR, 'B1_256', model='distilbert-base-uncased | bce loss', max_len=256, loss='bce', seed=42,
                val_macro_AP=round(macro_ap(Yva, P), 4), val_tuned_macro_F1=round(rep.F1.mean(), 4), best_epoch=3,
                notes='recovery run: resumed from epoch-1 checkpoint with new optimizer')
T = pd.DataFrame(rows)
print('\nOverall (validation):'); print(T.to_string(index=False))
A = pd.DataFrame(per_ap).round(4)
print('\nAverage Precision per label:'); print(A)
T.to_csv(f'{PROJECT_DIR}/results/tables/T08_baselines_vs_B1.csv', index=False)
A.to_csv(f'{PROJECT_DIR}/results/tables/T08_per_label_AP.csv')
print('CELL 4 DONE')